# Week 3, day 1 (afternoon) — Worksheet 16 SOLUTIONS: joining DataFrames

Executed in the lab image (pandas 3.0.5) against the real superstore extract.
Every quoted number is what it actually printed.

Questions 7 and 8 are the pair to remember, and the result is not the one most
people predict: the same join gives a **correct** revenue total and a **wrong**
count, from the same 837 rows.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Worksheet 16 — Joining DataFrames. Run this once.
import glob
import pandas as pd

BRONZE = "data/bronze/"

# The sales feed arrives one file per year. Stack them into one frame.
orders_raw = pd.concat([pd.read_csv(p) for p in sorted(glob.glob(BRONZE + "orders_*.csv"))],
                       ignore_index=True)
customers = pd.read_csv(BRONZE + "customers.csv")
products = pd.read_csv(BRONZE + "products.csv")
returns = pd.read_csv(BRONZE + "returns.csv")

print("orders_raw", orders_raw.shape, " <- as delivered; question 2 checks it")
print("customers", customers.shape)
print("products ", products.shape)
print("returns  ", returns.shape)

PART A — what am I joining?

### Question 1

Print the columns of all four tables, and for each one the column(s) it could be joined on. Note which key each table shares with `orders`.

In [ ]:
for name, df in [("orders", orders_raw), ("customers", customers),
                 ("products", products), ("returns", returns)]:
    print("%-10s %-9s %s" % (name, str(df.shape), list(df.columns)))
print()
for key in ("CustomerID", "ProductID", "OrderID", "LineID"):
    holders = [n for n, d in [("orders", orders_raw), ("customers", customers),
                              ("products", products), ("returns", returns)]
               if key in d.columns]
    print("  %-11s appears in: %s" % (key, ", ".join(holders)))

```
orders     (8100, 13) ['LineID', 'OrderID', 'ProductID', 'CustomerID', 'OrderDate', ...]
customers  (1832, 5)  ['CustomerID', 'CustomerName', 'Province', 'Region', 'CustomerSegment']
products   (1234, 6)  ['ProductID', 'ProductName', 'ProductCategory', ...]
returns    (558, 2)   ['OrderID', 'Status']

  CustomerID  appears in: orders, customers
  ProductID   appears in: orders, products
  OrderID     appears in: orders, returns
  LineID      appears in: orders
```

Three keys, three joins, and **`orders` is the only table that touches all of
them**. That shape has a name: `orders` is the *fact* table, `customers` and
`products` are *dimensions*, and every join radiates out from the middle. It is
the star schema, and week 3 day 4 builds one deliberately.

Two things to notice now.

**`LineID` appears in `orders` only.** No other table references it, which
already tells you it identifies a row rather than linking to anything.

**`returns` joins on `OrderID`, not `LineID`.** That is a different key from the
one that identifies an `orders` row, and it is the entire subject of questions 7
and 8. A table that joins on a *coarser* key than your fact's grain is the setup
for a fan-out.

Listing the keys before writing any `merge` costs thirty seconds. It is how you
notice that one of your three joins is not like the other two.

### Question 2

Before joining anything, establish the **grain** of each table. For each, print the row count, and whether its apparent key is unique.
> **NOTE:** a join multiplies rows whenever the key repeats on the side you join *to*. Find out which keys repeat before you find out the hard way.

In [ ]:
checks = [("orders_raw", orders_raw, "LineID"),
          ("orders_raw", orders_raw, "OrderID"),
          ("customers", customers, "CustomerID"),
          ("products", products, "ProductID"),
          ("returns", returns, "OrderID")]
for name, df, key in checks:
    print("  %-10s %-12s rows %5d   distinct %5d   unique %s"
          % (name, key, len(df), df[key].nunique(), df[key].is_unique))
print()
print("LineID is supposed to identify a line. It does not:",
      int(orders_raw.LineID.duplicated().sum()), "duplicated rows")
orders = orders_raw.drop_duplicates(subset="LineID").copy()
print("orders after dropping them:", len(orders), "rows -- use THIS from here on")
print()
print("orders is one row per LINE; returns is one row per ORDER.")
print("rows per OrderID in orders:")
print(orders.groupby("OrderID").size().value_counts().sort_index().to_string())

```
  orders_raw LineID       rows  8100   distinct  8060   unique False
  orders_raw OrderID      rows  8100   distinct  5361   unique False
  customers  CustomerID   rows  1832   distinct  1832   unique True
  products   ProductID    rows  1234   distinct  1234   unique True
  returns    OrderID      rows   558   distinct   558   unique True

LineID is supposed to identify a line. It does not: 40 duplicated rows
orders after dropping them: 8060 rows -- use THIS from here on
```

Two separate findings, and only one of them is a defect.

**`LineID` is not unique — 40 duplicated rows.** That *is* a defect. `LineID` is
the primary key of the source table, so a duplicate means the same line arrived
twice. Worksheet 17 finds out why (the 2012 file carries a re-delivery of
2011's last few days) and what a landing zone is supposed to do about it. Here,
drop them: 8,100 becomes **8,060**.

Notice how quietly that mattered. Every `SUM(Sales)` computed before this line
was inflated by 40 rows, and nothing would have flagged it.

**`OrderID` is not unique — 5,361 distinct across 8,060 rows.** That is **not** a
defect. It is the grain:

```
rows per OrderID in orders:
1    3341
2    1472
3     433
4     100
5      14
6       1
```

3,341 orders have one line; one order has six. `orders` is one row per order
*line*, exactly as a sales table should be.

The dimensions are clean — `CustomerID` and `ProductID` unique — which is what
makes joining to them safe. `returns.OrderID` is unique too, one row per returned
order.

**Grain first, joins second.** Everything on the rest of this sheet follows from
these five lines.

PART B — the safe join: many-to-one

### Question 3

Join `orders` to `customers` on `CustomerID` with `how="left"` and `validate="many_to_one"`. Print the row count before and after, and the new columns.
> **NOTE:** `validate=` makes pandas check your assumption instead of you hoping. Use it on every fact-to-dimension join.

In [ ]:
orders = orders_raw.drop_duplicates(subset="LineID")
before = len(orders)
enriched = orders.merge(customers, on="CustomerID", how="left",
                        validate="many_to_one")
print("orders before:", before)
print("orders after: ", len(enriched))
print("rows added:   ", len(enriched) - before)
print()
print("columns gained:", [c for c in enriched.columns if c not in orders.columns])
print()
print(enriched[["OrderID", "CustomerID", "CustomerName", "Region",
                "CustomerSegment"]].head(3).to_string(index=False))

```
orders before: 8060
orders after:  8060
rows added:    0

columns gained: ['CustomerName', 'Province', 'Region', 'CustomerSegment']

 OrderID  CustomerID   CustomerName   Region CustomerSegment
   13729    45703891 Matt Collister   Quebec     Home Office
   28774    48962730 Jessica Myrick  Ontario  Small Business
    9285    80357374 David Philippe Atlantic        Consumer
```

**Zero rows added.** Four columns of customer context attached to every order
line, and the table is exactly as long as it was.

That is what a **many-to-one** join is: many order lines, one customer each. The
fact table keeps its grain and gains attributes. Every join in a well-built
pipeline should look like this, and `rows added: 0` is how you prove it did.

`validate="many_to_one"` is the part worth adopting as a habit. It asserts that
`CustomerID` is unique on the **right** side, and raises immediately if it is
not. Without it, a duplicated customer row silently doubles the affected order
lines and inflates every total by a small, plausible, unnoticeable amount.

Three notes on the arguments:

**`how="left"`, not the default.** `merge`'s default is `how="inner"`, which
would silently drop any order line whose customer is missing. Question 5 shows
there are none here — but you want to *discover* that, not assume it.

**The join is on a shared column name**, so `on="CustomerID"` suffices. Question
9 handles the case where the names differ.

**`Region` came from `customers`.** Keep that in mind for question 9, where a
column of the same name exists on both sides.

### Question 4

Do the same for `products`, then compare the four `how=` values on the customer join: print the row count for `inner`, `left`, `right` and `outer`.

In [ ]:
orders = orders_raw.drop_duplicates(subset="LineID")
enriched = orders.merge(products, on="ProductID", how="left",
                        validate="many_to_one")
print("orders -> products, left:", len(enriched), "rows (was %d)" % len(orders))
print()
print("orders -> customers, by how=:")
for how in ("inner", "left", "right", "outer"):
    j = orders.merge(customers, on="CustomerID", how=how)
    print("  %-6s %6d rows" % (how, len(j)))
print()
print("orders rows:", len(orders), "| customers rows:", len(customers))

```
orders -> products, left: 8060 rows (was 8060)

orders -> customers, by how=:
  inner    8060 rows
  left     8060 rows
  right    8080 rows
  outer    8080 rows

orders rows: 8060 | customers rows: 1832
```

The products join behaves identically — 8,060 in, 8,060 out.

The four `how=` values split into two pairs, and the split tells you something
about the data:

**`inner` and `left` both give 8,060.** They agree only when every left row finds
a match. So this is evidence — not proof by assumption — that **every order line
has a customer**.

**`right` and `outer` both give 8,080**, twenty more. Those twenty are customers
with no orders, appearing once each with every order column null. Question 5
confirms it.

Which to use is a question about *what the row means*, not about which is safer:

| | keeps | use when |
|---|---|---|
| `inner` | only matched rows | you want the intersection, and you have said so |
| `left` | every fact row | the left table is your spine — **the usual answer** |
| `right` | every dimension row | rarely; flip the operands instead |
| `outer` | everything | reconciliation — finding what is on one side only |

**`left` is the default you should have in your head**, because in a pipeline the
fact table defines the row count and nothing should silently change it. `inner`
is a filter, and a filter you did not write down is a filter you will forget.

The 20-row gap between `left` and `outer` is the whole story of this dataset's
referential integrity, and it costs one line to see.

### Question 5

`right` and `outer` produced more rows than `orders` has. Find out why: redo the left join with `indicator=True` and print the counts of `_merge`, then do the same from the customers side.

In [ ]:
orders = orders_raw.drop_duplicates(subset="LineID")
j = orders.merge(customers, on="CustomerID", how="left", indicator=True)
print("orders LEFT customers:")
print(j._merge.value_counts().to_string())
print()
back = customers.merge(orders[["CustomerID"]].drop_duplicates(),
                       on="CustomerID", how="left", indicator=True)
print("customers LEFT orders:")
print(back._merge.value_counts().to_string())
print()
print("customers that never appear in orders:",
      int((back._merge == "left_only").sum()))

```
orders LEFT customers:
_merge
both          8060
left_only        0
right_only       0

customers LEFT orders:
_merge
both          1812
left_only       20
right_only       0

customers that never appear in orders: 20
```

`indicator=True` adds a `_merge` column recording, per row, where it came from —
`both`, `left_only`, or `right_only`. It turns "did the join work?" into a
`value_counts()`.

From the orders side: **8,060 `both`, zero `left_only`.** Every order line
matched a customer. The referential integrity is perfect in that direction, and
now you know rather than hope.

From the customers side: **1,812 matched, 20 `left_only`.** Twenty customers in
the CRM have never placed an order.

That asymmetry is the normal shape of a fact-to-dimension relationship, and it is
worth naming: a dimension may contain rows the fact never references, but a fact
row referencing a missing dimension row is a **defect**. The first is a customer
who has not bought yet; the second is a broken key.

So the two checks mean different things:

- `left_only` from the **fact** side → broken pipeline. Should be zero.
- `left_only` from the **dimension** side → a business fact. Count it and move on.

`indicator=True` costs nothing and answers both. Run it on any join whose
matching you have not previously verified — and delete the column afterwards, or
it follows you into the next `groupby`.

### Question 6

Those unmatched customers are a real business question. Isolate them with an **anti-join** and print how many there are, plus their segment breakdown.
> **NOTE:** pandas has no `anti-join`. The idiom is a left join with `indicator=True`, then filter to `left_only`.

In [ ]:
orders = orders_raw.drop_duplicates(subset="LineID")
sold_to = orders[["CustomerID"]].drop_duplicates()
never = (customers.merge(sold_to, on="CustomerID", how="left", indicator=True)
                  .query("_merge == 'left_only'")
                  .drop(columns=["_merge"]))
print("customers who have never ordered:", len(never))
print()
print(never.CustomerSegment.value_counts().rename("customers").to_string())
print()
print(never[["CustomerID", "CustomerName", "Region",
             "CustomerSegment"]].head(4).to_string(index=False))

```
customers who have never ordered: 20

CustomerSegment
Consumer          8
Corporate         4
Home Office       4
Small Business    4

 CustomerID   CustomerName  Region CustomerSegment
   10250297  Vivian Mathis  Quebec       Corporate
   18262264 Bradley Nguyen  Quebec       Corporate
   19637851 Anne McFarland   Yukon        Consumer
   25695577  Sally Hughsby Ontario       Home Office
```

Twenty named customers who have never bought anything — and that is a **sales
lead list**, produced by a join.

SQL has `NOT EXISTS` and `LEFT JOIN ... WHERE x IS NULL`. pandas has no
`how="anti"`, so the idiom is three steps:

```python
customers.merge(sold_to, on="CustomerID", how="left", indicator=True)
         .query("_merge == 'left_only'")
         .drop(columns=["_merge"])
```

Two details make it correct rather than nearly correct.

**`sold_to` is `drop_duplicates()`d first.** Joining against the raw
`orders.CustomerID` column would match each customer once per order line — the
fan-out from question 7, arriving early. Reduce the right side to distinct keys
before an existence check.

**Only the key column is carried.** `orders[["CustomerID"]]` rather than the whole
frame, so nothing else can collide or come along.

The segment breakdown is the part a business would act on: **8 of 20 are
Consumer**, twice any other segment. Whether that is a signal or noise on a
sample of twenty is exactly the caution from the ratio work elsewhere in this
course — but it is the right shape of question to ask next.

The mirror image is also worth running: products never sold, orders with no
matching customer, returns with no matching order. Each is one anti-join, and
each is either a lead list or a bug.

PART C — the join that costs money

### Question 7

Join `returns` to `orders` on `OrderID`, then answer *"how many orders were returned?"* the obvious way — `len()` on the result. Print the row count of each input, the joined row count, and the distinct `OrderID` count.
> **NOTE:** re-read question 2. `returns` has 558 rows. Predict the joined row count before running it.

In [ ]:
orders = orders_raw.drop_duplicates(subset="LineID")
r = orders.merge(returns, on="OrderID", how="inner")
print("orders rows: ", len(orders))
print("returns rows:", len(returns))
print("joined rows: ", len(r))
print()
print("COUNT(*) says returned orders:", len(r), "  <- wrong")
print("truth (returns rows):         ", len(returns))
print("distinct OrderID in result:   ", r.OrderID.nunique())
print()
print("overstated by %.0f%%" % (100 * (len(r) / len(returns) - 1)))
print("rows per returned order: %.2f" % (len(r) / len(returns)))

```
orders rows:  8060
returns rows: 558
joined rows:  837

COUNT(*) says returned orders: 837   <- wrong
truth (returns rows):          558
distinct OrderID in result:    558

overstated by 50%
rows per returned order: 1.50
```

**558 returns went in and 837 rows came out.** Asking "how many orders were
returned?" as `len(result)` gives **837**, and the true answer is **558** —
overstated by 50%.

The mechanism is question 2, arriving. `returns` has one row per order.
`orders` has 1.5 rows per order on average, because an order has several lines.
So each return row matches every line of its order and is duplicated once per
line.

The direction matters and is easy to get backwards: **`orders` did not fan out —
`returns` did.** Each order line still appears at most once. It is the 558
returns that became 837.

Notice how unhelpful the output is at telling you this. 837 is a plausible
number. It is stable, it will grow sensibly month over month, and it sits
between the two input row counts, so it does not even look extreme.

`distinct OrderID in result: 558` is the tell, and it is the check to run:

> After joining, is the count of the key you care about still what it was?

`r.OrderID.nunique()` is 558, matching `returns` exactly — so the *rows*
multiplied while the *orders* did not. Whenever those two disagree, `COUNT(*)` is
no longer counting the thing you named.

Now the part most people get wrong when they discover this. The instinct is
"the join is broken, every number from it is suspect." Question 8 shows that is
not true either.

### Question 8

Now sum `Sales` over that same join, and compare it with flagging returns **without** a join — use `isin` to add a boolean column and sum from `orders` directly. Print both totals. Explain why the count was wrong and the sum is not.
> **NOTE:** ask which table each number comes from. `Sales` lives on `orders`; the row count came from `returns`.

In [ ]:
orders = orders_raw.drop_duplicates(subset="LineID")
flagged = orders.copy()
flagged["IsReturned"] = flagged.OrderID.isin(set(returns.OrderID))
print("rows after isin flag:", len(flagged), "(orders had %d)" % len(orders))
print()
r = orders.merge(returns, on="OrderID", how="inner")
print("returned LINE rows (isin): ", int(flagged.IsReturned.sum()))
print("returned LINE rows (merge):", len(r))
print("returned orders:           ", flagged.loc[flagged.IsReturned, "OrderID"].nunique())
print()
print("SUM(Sales) via merge: %14.2f" % r.Sales.sum())
print("SUM(Sales) via isin:  %14.2f" % flagged.loc[flagged.IsReturned, "Sales"].sum())
print("identical:", round(r.Sales.sum(), 2)
      == round(flagged.loc[flagged.IsReturned, "Sales"].sum(), 2))
print()
print("SUM(Sales) total:     %14.2f" % flagged.Sales.sum())
print("returned share: %.1f%%"
      % (100 * flagged.loc[flagged.IsReturned, "Sales"].sum() / flagged.Sales.sum()))

```
rows after isin flag: 8060 (orders had 8060)

returned LINE rows (isin):  837
returned LINE rows (merge): 837
returned orders:            558

SUM(Sales) via merge: 1485707.73
SUM(Sales) via isin:  1485707.73
identical: True

SUM(Sales) total:    13570810.63
returned share: 10.9%
```

**The two agree to the cent.** `SUM(Sales)` over the fanned-out join is
`1485707.73`, and so is the total computed without any join at all.

So the join that was wrong in question 7 is **right** here — and the reason is
the single most useful idea on this sheet:

> A fan-out damages a measure only if the measure comes from the side that
> fanned out.

`Sales` lives on `orders`, at line grain. Each order line appears exactly **once**
in the join, because `returns.OrderID` is unique — one return row per order, so
no line can be matched twice. The `Sales` column is therefore neither duplicated
nor dropped, and summing it is safe.

The row *count* came from the other side. `returns` was duplicated 558 → 837, so
anything counting returns is inflated by exactly that ratio.

Same join, same 837 rows, one correct number and one wrong one. Which is why
"is this join safe?" is not a question with a yes/no answer — the honest question
is **"safe for which column?"**

The `isin` version is still the better code:

```python
orders["IsReturned"] = orders.OrderID.isin(set(returns.OrderID))
```

Not because the merge is wrong, but because it **cannot** go wrong. The row count
is unchanged by construction, so no later reader has to reason about fan-out to
trust a total. When you need a flag rather than columns, flag — do not join.

And `10.9%` of revenue returned is now a number you can defend, which is the
point of the whole exercise.

PART D — keys that do not line up

### Question 9

Two practical problems. First, join when the key has different names on each side: rename `customers.CustomerID` to `cust_id` and merge with `left_on`/`right_on`. Second, show what happens to a column name that exists on both sides — merge `orders` with `customers` when both carry `Region`.

In [ ]:
orders = orders_raw.drop_duplicates(subset="LineID")
renamed = customers.rename(columns={"CustomerID": "cust_id"})
j = orders.merge(renamed, left_on="CustomerID", right_on="cust_id", how="left")
print("left_on/right_on ->", len(j), "rows")
print("both key columns survive:",
      [c for c in j.columns if c in ("CustomerID", "cust_id")])
print()
o2 = orders.copy()
o2["Region"] = "FROM-ORDERS"
clash = o2.merge(customers, on="CustomerID", how="left")
print("colliding column names became:",
      [c for c in clash.columns if c.startswith("Region")])
print()
named = o2.merge(customers, on="CustomerID", how="left",
                 suffixes=("_order", "_customer"))
print("with suffixes=:", [c for c in named.columns if c.startswith("Region")])

```
left_on/right_on -> 8060 rows
both key columns survive: ['CustomerID', 'cust_id']

colliding column names became: ['Region_x', 'Region_y']

with suffixes=: ['Region_order', 'Region_customer']
```

Two everyday annoyances, both with a real trap inside.

**Different key names.** `left_on="CustomerID", right_on="cust_id"` joins fine —
and **both key columns survive** in the output, holding identical values. That is
a redundant column that will follow you into every downstream step, and worse, it
is the kind of thing that makes a later `drop_duplicates()` or a column-count
assertion behave unexpectedly. Drop one, or rename before merging so you can use
plain `on=`.

**Colliding names.** When both sides carry `Region`, pandas silently renames them
`Region_x` and `Region_y`. Nothing errors, and now the least informative column
names in your pipeline are on the two columns most likely to be confused.

`suffixes=("_order", "_customer")` fixes it, and the fix is not cosmetic — it is
the difference between a reader knowing which `Region` they are grouping by and
guessing. On this data the two genuinely differ in meaning: an order's region
and the customer's home region are not the same fact, and a report grouped by the
wrong one is wrong in a way no check will catch.

**The habit that avoids both:** select the columns you want from the right side
before merging.

```python
orders.merge(customers[["CustomerID", "CustomerSegment"]],
             on="CustomerID", how="left", validate="many_to_one")
```

You cannot collide with a column you did not bring, and the merge documents what
it is actually for.

### Question 10

Finally, assert the assumption that would have prevented question 7: merge `orders` with `returns` on `OrderID` using `validate="one_to_one"`. **This is supposed to fail.** Read the message and connect it to question 2.

In [ ]:
orders = orders_raw.drop_duplicates(subset="LineID")
print("distinct OrderID in orders: ", orders.OrderID.nunique(),
      "over", len(orders), "rows")
print("distinct OrderID in returns:", returns.OrderID.nunique(),
      "over", len(returns), "rows")
print()
print(orders.merge(returns, on="OrderID", validate="one_to_one").shape)

```
distinct OrderID in orders:  5361 over 8060 rows
distinct OrderID in returns: 558 over 558 rows

MergeError: Merge keys are not unique in left dataset; not a one-to-one merge
Duplicates in left:
  OrderID
   37537
   37537
   44069
...
```

`validate="one_to_one"` asserts the key is unique on **both** sides. It is not —
`orders` has 5,361 distinct `OrderID` across 8,060 rows — so pandas refuses
before producing a single row, and names the offending keys.

That is question 7's fan-out, caught in advance by one keyword argument.

The four values are four different claims:

| value | you are asserting |
|---|---|
| `"one_to_one"` | unique on **both** sides |
| `"one_to_many"` | unique on the left, repeated on the right |
| `"many_to_one"` | repeated on the left, unique on the right |
| `"many_to_many"` | nothing — the default |

**`many_to_one` is the one to reach for daily.** Every fact-to-dimension join in
this sheet is one, and stating it converts the most common silent pipeline bug —
a duplicated dimension row quietly inflating totals by a few percent — into an
immediate, named failure.

It is worth being precise about what `validate=` does and does not buy you. It
would **not** have saved question 7: `many_to_one` on `orders`-to-`returns`
passes, because `returns.OrderID` really is unique. The fan-out there is on the
returns side and is perfectly legitimate as a join — the error was asking it to
count. **`validate=` checks the shape of the join. It cannot check the meaning of
your question.**

**What this sheet established:**

| | |
|---|---|
| the keys | 3 joins from one fact table — `orders` is the only table touching all of them |
| the grain | `LineID` **not unique — 40 duplicate rows**, silently inflating every total until dropped |
| | `OrderID` repeats legitimately: 5,361 orders across 8,060 lines |
| a safe join | `many_to_one` + `how="left"` → **8,060 in, 8,060 out, 0 added** |
| `how=` | `inner`/`left` 8,060; `right`/`outer` **8,080** — the gap is the data telling you something |
| `indicator=True` | 0 unmatched from the fact side; **20** customers who never ordered |
| the anti-join | those 20, with a segment breakdown — a lead list, not a bug |
| the trap | 558 returns → **837 rows**; `COUNT(*)` overstates by **50%** |
| the twist | `SUM(Sales)` is **identical either way** — 1,485,707.73 — because `Sales` did not fan out |

The rule to carry: **a join is safe for a column, not in general.** Ask which
side each number comes from, and whether that side was duplicated.

Worksheet 17 puts these joins to work, building a bronze → silver → gold
pipeline on the same four files — including finding out where those 40 duplicate
rows came from.